## 0. Obiettivo del notebook

Questo notebook è costruito per produrre in modo riproducibile:

- i grafici per il **Capitolo 6**;
- lo schema logico del **Capitolo 7**;
- le metriche, le tabelle e i grafici del **Capitolo 8**;
- i materiali di supporto per **Capitoli 9 e 10**.

La pipeline segue la struttura della repository `iot-audit`, con preprocessing, EDA, training supervisionato e benchmark separati per task binario e multiclass.

In [1]:
from __future__ import annotations

import time
from pathlib import Path
from typing import List, Tuple

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    auc,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, label_binarize, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

HAS_XGB = True
HAS_LGBM = True

try:
    from xgboost import XGBClassifier
except Exception as exc:
    HAS_XGB = False
    XGBClassifier = None
    print(f"[WARN] xgboost non disponibile: {exc}")

try:
    from lightgbm import LGBMClassifier
except Exception as exc:
    HAS_LGBM = False
    LGBMClassifier = None
    print(f"[WARN] lightgbm non disponibile: {exc}")

In [2]:
# =========================
# CONFIGURAZIONE GENERALE
# =========================
RANDOM_STATE = 42
TEST_SIZE = 0.20
TOP_MISSING = 20
HIGH_CARDINALITY_THRESHOLD = 50

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "train_test_network.csv"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

for folder in [FIGURES_DIR, TABLES_DIR, MODELS_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "savefig.dpi": 300,
})

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")

Project root: C:\Users\emanu\OneDrive\Desktop\iot-audit
Dataset path: C:\Users\emanu\OneDrive\Desktop\iot-audit\data\train_test_network.csv


## 1. Caricamento e controllo iniziale del dataset

In [3]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset non trovato in: {DATA_PATH}. "
        "Verifica di aver copiato train_test_network.csv nella cartella data/."
    )

df_raw = pd.read_csv(DATA_PATH)
print("Shape dataset:", df_raw.shape)
display(df_raw.head(3))

Shape dataset: (211043, 44)


,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,-,290.371539,101568,2592,OTH,...,0,0,-,-,-,-,-,-,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000102,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000148,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor


In [4]:
print("Colonne disponibili:")
print(df_raw.columns.tolist())

print("\nTipi di dato:")
display(df_raw.dtypes.value_counts())

print("\nInformazioni sintetiche:")
df_raw.info()

Colonne disponibili:
['src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_query', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth', 'http_method', 'http_uri', 'http_version', 'http_request_body_len', 'http_response_body_len', 'http_status_code', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_addl', 'weird_notice', 'label', 'type']

Tipi di dato:


str        27
int64      16
float64     1
Name: count, dtype: int64


Informazioni sintetiche:
<class 'pandas.DataFrame'>
RangeIndex: 211043 entries, 0 to 211042
Data columns (total 44 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   src_ip                  211043 non-null  str    
 1   src_port                211043 non-null  int64  
 2   dst_ip                  211043 non-null  str    
 3   dst_port                211043 non-null  int64  
 4   proto                   211043 non-null  str    
 5   service                 211043 non-null  str    
 6   duration                211043 non-null  float64
 7   src_bytes               211043 non-null  int64  
 8   dst_bytes               211043 non-null  int64  
 9   conn_state              211043 non-null  str    
 10  missed_bytes            211043 non-null  int64  
 11  src_pkts                211043 non-null  int64  
 12  src_ip_bytes            211043 non-null  int64  
 13  dst_pkts                211043 non-null  int64  
 14  dst_i

In [5]:
summary_quality = pd.DataFrame({
    "missing_values": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().sum() / len(df_raw) * 100).round(2),
    "n_unique": df_raw.nunique(dropna=True),
}).sort_values("missing_values", ascending=False)

display(summary_quality.head(20))
print(f"Duplicati: {df_raw.duplicated().sum()}")

,missing_values,missing_pct,n_unique
src_ip,0,0.0,51
src_port,0,0.0,26628
dst_ip,0,0.0,753
dst_port,0,0.0,2039
proto,0,0.0,3
service,0,0.0,9
duration,0,0.0,68570
src_bytes,0,0.0,2199
dst_bytes,0,0.0,2338
conn_state,0,0.0,13


Duplicati: 20569


## 2. Preparazione di base e pulizia controllata

La repository elimina o non utilizza colonne testuali ad alta cardinalità che potrebbero introdurre leakage o aumentare in modo artificiale la dimensionalità del problema.

In [6]:
TARGET_BINARY = "label"
TARGET_MULTICLASS = "type"

manual_drop_candidates = [
    "src_ip", "dst_ip",
    "http_uri", "ssl_subject", "ssl_issuer",
    "dns_query", "http_host", "user_agent",
]

def prepare_base_dataframe(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    df = df.copy()

    # Normalizza placeholder testuali comuni
    df = df.replace(["-", "NA", "N/A", ""], np.nan)

    # Selezione colonne da eliminare: manuali + alta cardinalità testuale
    drop_cols = [c for c in manual_drop_candidates if c in df.columns]

    text_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    text_cols = [c for c in text_cols if c not in [TARGET_BINARY, TARGET_MULTICLASS]]

    for col in text_cols:
        nunique = df[col].nunique(dropna=True)
        if nunique > HIGH_CARDINALITY_THRESHOLD and col not in drop_cols:
            drop_cols.append(col)

    # Colonne costanti
    constant_cols = [
        c for c in df.columns
        if c not in [TARGET_BINARY, TARGET_MULTICLASS] and df[c].nunique(dropna=True) <= 1
    ]
    drop_cols.extend([c for c in constant_cols if c not in drop_cols])

    # Elimina duplicati
    df = df.drop_duplicates()

    # Drop finale
    existing_drop_cols = [c for c in drop_cols if c in df.columns]
    df = df.drop(columns=existing_drop_cols, errors="ignore")

    return df, existing_drop_cols

df, dropped_columns = prepare_base_dataframe(df_raw)

print("Colonne eliminate:", dropped_columns)
print("Shape dopo pulizia:", df.shape)
display(df.head(3))

Colonne eliminate: ['src_ip', 'dst_ip', 'http_uri', 'ssl_subject', 'ssl_issuer', 'dns_query', 'http_version', 'weird_notice']
Shape dopo pulizia: (190474, 36)


,src_port,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,...,http_request_body_len,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,label,type
0,4444,49178,tcp,NaN,290.371539,101568,2592,OTH,0,108,...,0,0,0,NaN,NaN,NaN,NaN,NaN,1,backdoor
1,49180,8080,tcp,NaN,0.000102,0,0,REJ,0,1,...,0,0,0,NaN,NaN,NaN,NaN,NaN,1,backdoor
2,49180,8080,tcp,NaN,0.000148,0,0,REJ,0,1,...,0,0,0,NaN,NaN,NaN,NaN,NaN,1,backdoor


## 3. Grafici per il Capitolo 6

In [7]:
def save_figure(path: Path) -> None:
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

def annotate_bars(ax):
    for p in ax.patches:
        height = p.get_height()
        ax.annotate(
            f"{int(height):,}".replace(",", "."),
            (p.get_x() + p.get_width() / 2, height),
            ha="center",
            va="bottom",
            fontsize=9,
            xytext=(0, 3),
            textcoords="offset points",
        )

def plot_distribution(series: pd.Series, title: str, xlabel: str, filename: str, rotate=0):
    counts = series.value_counts(dropna=False)
    fig, ax = plt.subplots()
    ax.bar(counts.index.astype(str), counts.values)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Numero di flussi")
    ax.tick_params(axis="x", rotation=rotate)
    annotate_bars(ax)
    save_figure(FIGURES_DIR / filename)
    plt.show()
    return counts

binary_counts = plot_distribution(
    df[TARGET_BINARY],
    "Distribuzione delle classi - Task binario",
    "Classe",
    "class_distribution_binary.png",
    rotate=0
)

multiclass_counts = plot_distribution(
    df[TARGET_MULTICLASS],
    "Distribuzione delle classi - Task multiclass",
    "Tipo di traffico / attacco",
    "class_distribution_multiclass.png",
    rotate=45
)

display(binary_counts.to_frame("count"))
display(multiclass_counts.to_frame("count").head(20))

,count
label,
1,148434
0,42040


,count
type,
normal,42040
scanning,20000
ddos,19993
injection,19964
password,19861
dos,18992
backdoor,18711
xss,15137
ransomware,14735


In [8]:
# Missing values: top 20 colonne
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if len(missing) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    missing.head(TOP_MISSING).plot(kind="bar", ax=ax)
    ax.set_title("Valori mancanti - Top colonne")
    ax.set_xlabel("Colonna")
    ax.set_ylabel("Numero di valori mancanti")
    ax.tick_params(axis="x", rotation=45)
    save_figure(FIGURES_DIR / "missing_values_top20.png")
    display(missing.head(TOP_MISSING).to_frame("missing_values"))
else:
    print("Nessun valore mancante rilevato.")

,missing_values
http_orig_mime_types,190458
weird_addl,190317
http_resp_mime_types,190270
http_method,190187
http_user_agent,190187
http_trans_depth,190171
weird_name,190118
ssl_version,190073
ssl_established,190073
ssl_resumed,190073


In [9]:
# Heatmap di correlazione sulle variabili numeriche
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()

    plt.figure(figsize=(14, 10))
    sns.heatmap(corr, cmap="coolwarm", center=0, square=False, cbar_kws={"shrink": 0.8})
    plt.title("Matrice di correlazione delle variabili numeriche")
    save_figure(FIGURES_DIR / "correlation_heatmap.png")
else:
    print("Numero insufficiente di colonne numeriche per la heatmap.")

In [10]:
binary_pct = (binary_counts / binary_counts.sum() * 100).round(2)
display(pd.DataFrame({"count": binary_counts, "percentage": binary_pct}))

,count,percentage
label,,
1,148434,77.93
0,42040,22.07


## 4. Funzioni di supporto per gli esperimenti

In [11]:
def infer_feature_types(X: pd.DataFrame, threshold: int = HIGH_CARDINALITY_THRESHOLD):
    # Identifica feature numeriche e categoriche a bassa cardinalità
    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = []

    for col in X.columns:
        if col in numeric_features:
            continue
        nunique = X[col].nunique(dropna=True)
        if nunique <= threshold:
            categorical_features.append(col)

    return numeric_features, categorical_features

def build_preprocessor(X: pd.DataFrame):
    numeric_features, categorical_features = infer_feature_types(X)

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )

    return preprocessor, numeric_features, categorical_features

def measure_inference_time_ms(pipeline: Pipeline, X_eval: pd.DataFrame, repeats: int = 10) -> float:
    sample_size = min(len(X_eval), 1000)
    X_sample = X_eval.sample(n=sample_size, random_state=RANDOM_STATE) if len(X_eval) > sample_size else X_eval.copy()

    _ = pipeline.predict(X_sample.iloc[: min(10, len(X_sample))])

    timings = []
    for _ in range(repeats):
        start = time.perf_counter()
        _ = pipeline.predict(X_sample)
        end = time.perf_counter()
        timings.append((end - start) * 1000.0)

    mean_ms = float(np.mean(timings))
    ms_per_1000 = mean_ms * (1000.0 / len(X_sample))
    return ms_per_1000

def save_model_and_get_size_mb(pipeline: Pipeline, filename: str) -> float:
    path = MODELS_DIR / filename
    joblib.dump(pipeline, path)
    return path.stat().st_size / (1024 ** 2)

def plot_confusion_matrix(y_true, y_pred, labels, title, filename, normalize=False):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    if normalize:
        cm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        fmt = ".2f"
    else:
        fmt = "d"

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt=fmt, cmap="Blues", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predetto")
    ax.set_ylabel("Reale")
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels, rotation=0)
    save_figure(filename)

def plot_metric_comparison(df_results: pd.DataFrame, metric_col: str, title: str, filename: str, sort_desc=True):
    plot_df = df_results[["model", metric_col]].copy()
    plot_df = plot_df.sort_values(metric_col, ascending=not sort_desc)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.bar(plot_df["model"], plot_df[metric_col])
    ax.set_title(title)
    ax.set_xlabel("Modello")
    ax.set_ylabel(metric_col.replace("_", " ").title())
    ax.tick_params(axis="x", rotation=20)
    annotate_bars(ax)
    save_figure(filename)

def get_models(task: str, n_classes: int = 2):
    models = {}

    models["Logistic Regression"] = LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE,
    )

    models["Random Forest"] = RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    if HAS_XGB:
        if task == "binary":
            models["XGBoost"] = XGBClassifier(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                eval_metric="logloss",
            )
        else:
            models["XGBoost"] = XGBClassifier(
                n_estimators=400,
                learning_rate=0.05,
                max_depth=6,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=1.0,
                random_state=RANDOM_STATE,
                n_jobs=-1,
                objective="multi:softprob",
                num_class=n_classes,
                eval_metric="mlogloss",
            )

    if HAS_LGBM:
        models["LightGBM"] = LGBMClassifier(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    return models

def evaluate_binary_pipeline(pipeline: Pipeline, X_test: pd.DataFrame, y_test: pd.Series, model_name: str):
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    result = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "pr_auc": average_precision_score(y_test, y_prob),
        "inference_ms_per_1k": measure_inference_time_ms(pipeline, X_test),
    }
    result["model_size_mb"] = save_model_and_get_size_mb(pipeline, f"binary_{model_name.lower().replace(' ', '_')}.joblib")
    return result, y_pred, y_prob

def evaluate_multiclass_pipeline(pipeline: Pipeline, X_test: pd.DataFrame, y_test: pd.Series, model_name: str):
    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)
    labels = list(pipeline.named_steps["model"].classes_)

    y_test_bin = label_binarize(y_test, classes=labels)
    if y_test_bin.shape[1] == 1:
        y_test_bin = np.hstack([1 - y_test_bin, y_test_bin])

    result = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "macro_f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "macro_precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "roc_auc_macro": roc_auc_score(y_test_bin, y_prob, average="macro", multi_class="ovr"),
        "pr_auc_macro": average_precision_score(y_test_bin, y_prob, average="macro"),
        "inference_ms_per_1k": measure_inference_time_ms(pipeline, X_test),
    }
    result["model_size_mb"] = save_model_and_get_size_mb(pipeline, f"multiclass_{model_name.lower().replace(' ', '_')}.joblib")
    return result, y_pred, y_prob, labels

## 5. Esperimento 1 — Classificazione binaria

In [12]:
binary_df = df.copy()
binary_df = binary_df.drop(columns=[TARGET_MULTICLASS], errors="ignore")

X_binary = binary_df.drop(columns=[TARGET_BINARY])
y_binary = binary_df[TARGET_BINARY]

if y_binary.dtype == object:
    y_binary = y_binary.astype(str)

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_binary, y_binary, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_binary
)

preprocessor_b, numeric_features_b, categorical_features_b = build_preprocessor(X_train_b)

print("Feature numeriche:", len(numeric_features_b))
print("Feature categoriche:", len(categorical_features_b))
print("Train shape:", X_train_b.shape, "Test shape:", X_test_b.shape)

Feature numeriche: 16
Feature categoriche: 18
Train shape: (152379, 34) Test shape: (38095, 34)


In [13]:
binary_models = get_models(task="binary")

binary_results = []
binary_predictions = {}
binary_probabilities = {}
binary_pipelines = {}

for model_name, estimator in binary_models.items():
    print(f"Training binary model: {model_name}")
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor_b),
        ("model", estimator),
    ])
    pipe.fit(X_train_b, y_train_b)

    result, y_pred, y_prob = evaluate_binary_pipeline(pipe, X_test_b, y_test_b, model_name)
    binary_results.append(result)
    binary_predictions[model_name] = y_pred
    binary_probabilities[model_name] = y_prob
    binary_pipelines[model_name] = pipe

binary_results_df = pd.DataFrame(binary_results).sort_values(by="f1", ascending=False)
display(binary_results_df)
binary_results_df.to_csv(OUTPUT_DIR / "binary_results.csv", index=False)

Training binary model: Logistic Regression
Training binary model: Random Forest
Training binary model: XGBoost
Training binary model: LightGBM
[LightGBM] [Info] Number of positive: 118747, number of negative: 33632
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.043618 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 152379, number of used features: 67
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.779287 -> initscore=1.261517
[LightGBM] [Info] Start training from score 1.261517


C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

,model,accuracy,precision,recall,f1,roc_auc,pr_auc,inference_ms_per_1k,model_size_mb
3,LightGBM,0.998897,0.999057,0.999528,0.999293,0.999989,0.999997,60.47482,1.372985
1,Random Forest,0.998504,0.998687,0.999394,0.999040,0.999978,0.999993,142.50073,22.161906
2,XGBoost,0.998477,0.998822,0.999225,0.999023,0.999974,0.999993,31.26554,0.754624
0,Logistic Regression,0.984355,0.990226,0.989692,0.989959,0.994496,0.998107,18.68740,0.012278


In [14]:
plot_metric_comparison(binary_results_df, "f1", "Confronto F1-score (Binary)", FIGURES_DIR / "binary_f1_comparison.png")
plot_metric_comparison(binary_results_df, "roc_auc", "Confronto ROC-AUC (Binary)", FIGURES_DIR / "binary_roc_auc_comparison.png")
plot_metric_comparison(binary_results_df, "pr_auc", "Confronto PR-AUC (Binary)", FIGURES_DIR / "binary_pr_auc_comparison.png")
plot_metric_comparison(binary_results_df, "inference_ms_per_1k", "Confronto latenza (ms / 1000 predizioni) - Binary", FIGURES_DIR / "binary_latency_comparison.png", sort_desc=False)
plot_metric_comparison(binary_results_df, "model_size_mb", "Confronto dimensione modello (MB) - Binary", FIGURES_DIR / "binary_size_comparison.png", sort_desc=False)

In [15]:
best_binary = binary_results_df.iloc[0]["model"]
print("Miglior modello binary:", best_binary)

plot_confusion_matrix(
    y_test_b,
    binary_predictions[best_binary],
    labels=sorted(pd.unique(y_binary).tolist()),
    title=f"Confusion Matrix Binary - {best_binary}",
    filename=FIGURES_DIR / "binary_confusion_matrix_best.png",
    normalize=False,
)

best_prob = binary_probabilities[best_binary]
fpr, tpr, _ = roc_curve(y_test_b, best_prob)
precision, recall, _ = precision_recall_curve(y_test_b, best_prob)
pr_auc_value = auc(recall, precision)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, label=f"{best_binary}")
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_title(f"ROC Curve Binary - {best_binary}")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
save_figure(FIGURES_DIR / "binary_roc_curve_best.png")

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall, precision, label=f"{best_binary} (AUC={pr_auc_value:.3f})")
ax.set_title(f"Precision-Recall Curve Binary - {best_binary}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
save_figure(FIGURES_DIR / "binary_pr_curve_best.png")

Miglior modello binary: LightGBM


## 6. Esperimento 2 — Classificazione multiclass

In [16]:
from sklearn.preprocessing import LabelEncoder

multiclass_df = df.copy()
multiclass_df = multiclass_df.drop(columns=[TARGET_BINARY], errors="ignore")

X_multi = multiclass_df.drop(columns=[TARGET_MULTICLASS])
y_multi = multiclass_df[TARGET_MULTICLASS].astype(str)


from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_multi_encoded = label_encoder.fit_transform(y_multi)


X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi,
    y_multi_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_multi_encoded
)

preprocessor_m, numeric_features_m, categorical_features_m = build_preprocessor(X_train_m)

print("Feature numeriche:", len(numeric_features_m))
print("Feature categoriche:", len(categorical_features_m))
print("Classi multiclass:", len(label_encoder.classes_))
print("Train shape:", X_train_m.shape, "Test shape:", X_test_m.shape)

Feature numeriche: 16
Feature categoriche: 18
Classi multiclass: 10
Train shape: (152379, 34) Test shape: (38095, 34)


In [17]:
multiclass_models = get_models(
    task="multiclass",
    n_classes=len(label_encoder.classes_)
)

multiclass_results = []
multiclass_predictions = {}
multiclass_probabilities = {}
multiclass_pipelines = {}
multiclass_labels = None

for model_name, estimator in multiclass_models.items():
    print(f"Training multiclass model: {model_name}")
    
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor_m),
        ("model", estimator),
    ])
    
    pipe.fit(X_train_m, y_train_m)

    result, y_pred, y_prob, labels = evaluate_multiclass_pipeline(
        pipe, X_test_m, y_test_m, model_name
    )

    y_pred_labels = label_encoder.inverse_transform(y_pred)

    multiclass_results.append(result)
    multiclass_predictions[model_name] = y_pred_labels
    multiclass_probabilities[model_name] = y_prob
    multiclass_pipelines[model_name] = pipe
    multiclass_labels = labels

Training multiclass model: Logistic Regression
Training multiclass model: Random Forest
Training multiclass model: XGBoost
Training multiclass model: LightGBM
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.036518 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2460
[LightGBM] [Info] Number of data points in the train set: 152379, number of used features: 67
[LightGBM] [Info] Start training from score -2.320389
[LightGBM] [Info] Start training from score -2.254157
[LightGBM] [Info] Start training from score -2.305536
[LightGBM] [Info] Start training from score -2.255596
[LightGBM] [Info] Start training from score -5.209092
[LightGBM] [Info] Start training from score -1.510893
[LightGBM] [Info] Start training from score -2.260744
[LightGBM] [Info] Start training from score -2.559289
[LightGBM] [Info] Start training from score -2.253782
[

C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

In [18]:
multiclass_results_df = pd.DataFrame(multiclass_results).sort_values(by="macro_f1", ascending=False)
display(multiclass_results_df)
multiclass_results_df.to_csv(OUTPUT_DIR / "multiclass_results.csv", index=False)

,model,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,roc_auc_macro,pr_auc_macro,inference_ms_per_1k,model_size_mb
2,XGBoost,0.990051,0.971537,0.990097,0.968533,0.974838,0.999817,0.985653,188.58947,9.478388
1,Random Forest,0.988109,0.968729,0.988169,0.965071,0.972807,0.999646,0.986477,191.64886,182.297221
3,LightGBM,0.989657,0.968610,0.989664,0.968307,0.968941,0.999823,0.983527,192.83189,13.124496
0,Logistic Regression,0.851765,0.788496,0.848802,0.808765,0.784068,0.985340,0.835874,10.92738,0.020144


In [19]:
plot_metric_comparison(multiclass_results_df, "accuracy", "Confronto Accuracy (Multiclass)", FIGURES_DIR / "multiclass_accuracy_comparison.png")
plot_metric_comparison(multiclass_results_df, "macro_f1", "Confronto Macro-F1 (Multiclass)", FIGURES_DIR / "multiclass_macro_f1_comparison.png")
plot_metric_comparison(multiclass_results_df, "weighted_f1", "Confronto Weighted F1 (Multiclass)", FIGURES_DIR / "multiclass_weighted_f1_comparison.png")
plot_metric_comparison(multiclass_results_df, "roc_auc_macro", "Confronto ROC-AUC Macro (Multiclass)", FIGURES_DIR / "multiclass_roc_auc_macro_comparison.png")
plot_metric_comparison(multiclass_results_df, "pr_auc_macro", "Confronto PR-AUC Macro (Multiclass)", FIGURES_DIR / "multiclass_pr_auc_macro_comparison.png")
plot_metric_comparison(multiclass_results_df, "inference_ms_per_1k", "Confronto latenza (ms / 1000 predizioni) - Multiclass", FIGURES_DIR / "multiclass_latency_comparison.png", sort_desc=False)
plot_metric_comparison(multiclass_results_df, "model_size_mb", "Confronto dimensione modello (MB) - Multiclass", FIGURES_DIR / "multiclass_size_comparison.png", sort_desc=False)

In [20]:
best_multi = multiclass_results_df.iloc[0]["model"]
print("Miglior modello multiclass:", best_multi)

y_test_labels = label_encoder.inverse_transform(y_test_m)

y_pred_labels = multiclass_predictions[best_multi]

plot_confusion_matrix(
    y_test_labels,
    y_pred_labels,
    labels=label_encoder.classes_,
    title=f"Confusion Matrix Multiclass - {best_multi}",
    filename=FIGURES_DIR / "multiclass_confusion_matrix_best.png",
    normalize=False,
)

report = classification_report(
    y_test_labels,
    y_pred_labels,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).T
display(report_df)
report_df.to_csv(OUTPUT_DIR / "multiclass_classification_report_best.csv")

Miglior modello multiclass: XGBoost


,precision,recall,f1-score,support
backdoor,1.000000,1.000000,1.000000,3742.000000
ddos,0.991184,0.983996,0.987577,3999.000000
dos,0.993631,0.985523,0.989560,3799.000000
injection,0.984002,0.970448,0.977178,3993.000000
mitm,0.773333,0.836538,0.803695,208.000000
normal,0.997029,0.997978,0.997504,8408.000000
password,0.996962,0.991440,0.994193,3972.000000
ransomware,0.998983,1.000000,0.999491,2947.000000
scanning,0.987605,0.996000,0.991785,4000.000000
xss,0.962605,0.986455,0.974384,3027.000000


In [21]:
per_class_f1 = report_df.loc[
    [c for c in report_df.index if c not in ["accuracy", "macro avg", "weighted avg"]],
    "f1-score"
].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(per_class_f1.index, per_class_f1.values)
ax.set_title(f"F1-score per classe - {best_multi}")
ax.set_xlabel("Classe")
ax.set_ylabel("F1-score")
ax.tick_params(axis="x", rotation=45)
annotate_bars(ax)
save_figure(FIGURES_DIR / "multiclass_per_class_f1_best.png")

## 7. Esportazione finale

I file generati possono essere inseriti direttamente nella tesi:

- `figures/` → grafici dei capitoli 6 e 8
- `outputs/` → tabelle e report
- `models/` → pipeline salvate per eventuali controlli successivi

In [22]:
print("File prodotti:")
for folder in [FIGURES_DIR, TABLES_DIR, MODELS_DIR, OUTPUT_DIR]:
    print(f"\n{folder.name}:")
    for p in sorted(folder.glob("*")):
        print(" -", p.name)

File prodotti:

figures:
 - binary_confusion_matrix_best.png
 - binary_f1_comparison.png
 - binary_latency_comparison.png
 - binary_pr_auc_comparison.png
 - binary_pr_curve_best.png
 - binary_roc_auc_comparison.png
 - binary_roc_curve_best.png
 - binary_size_comparison.png
 - class_distribution_binary.png
 - class_distribution_multiclass.png
 - correlation_heatmap.png
 - missing_values_top20.png
 - multiclass_accuracy_comparison.png
 - multiclass_confusion_matrix_best.png
 - multiclass_latency_comparison.png
 - multiclass_macro_f1_comparison.png
 - multiclass_per_class_f1_best.png
 - multiclass_pr_auc_macro_comparison.png
 - multiclass_roc_auc_macro_comparison.png
 - multiclass_size_comparison.png
 - multiclass_weighted_f1_comparison.png

tables:

models:
 - binary_lightgbm.joblib
 - binary_logistic_regression.joblib
 - binary_random_forest.joblib
 - binary_xgboost.joblib
 - multiclass_lightgbm.joblib
 - multiclass_logistic_regression.joblib
 - multiclass_random_forest.joblib
 - multic